# Type Hints

Type hints let you annotate variables, function parameters, and return values with their expected types. Python ignores them at runtime, but tools like Pylance (VS Code) and mypy read them to catch type errors before you run the code, catching the same category of mistakes that compiled languages catch at compile time. They also make code significantly easier to read and navigate.

**What's inside:** variable and function annotations, built-in generics (`list[int]`, `dict[str, float]`), `Optional`, `Union`, `Callable`, `TypeVar` for generic functions, class annotations with `ClassVar`, and `Protocol` for structural subtyping.

**Learn more:** [typing module](https://docs.python.org/3/library/typing.html)

## 1. Variable Annotations

Annotations are stored in `__annotations__` but have no runtime effect.

In [ ]:
x: int = 5
name: str = 'Alice'
pi: float = 3.14
active: bool = True

print(x, name, pi, active)

In [ ]:
# annotation without assignment (declares intent, no value yet)
count: int

# annotations are stored but not enforced
x: int = 'oops'   # no error at runtime
print(x)

## 2. Function Annotations

Annotate parameters with `: type` and the return value with `-> type`.

In [ ]:
def greet(name: str) -> str:
    return f'Hello, {name}!'

greet('Alice')

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

add(3, 4)

In [ ]:
# -> None for functions that don't return a value
def log(message: str) -> None:
    print(f'[LOG] {message}')

log('started')

In [ ]:
# annotations are accessible at runtime via __annotations__
greet.__annotations__

## 3. Built-in Generics

Python 3.9+ supports generic types directly using built-in classes. No import needed.

In [ ]:
# list, dict, set, tuple with element types
def total(scores: list[int]) -> int:
    return sum(scores)

total([10, 20, 30])

In [ ]:
# dict with key and value types
def word_lengths(words: list[str]) -> dict[str, int]:
    return {w: len(w) for w in words}

word_lengths(['apple', 'banana', 'fig'])

In [ ]:
# tuple (fixed length, each position typed)
def min_max(nums: list[float]) -> tuple[float, float]:
    return min(nums), max(nums)

min_max([3.0, 1.5, 4.2, 2.8])

In [ ]:
# set
def unique_ids(items: list[int]) -> set[int]:
    return set(items)

unique_ids([1, 2, 2, 3, 3, 3])

## 4. Optional and Union

Use when a value can be one of several types, or can be absent (`None`).

In [ ]:
# X | None means the value can be X or None (Python 3.10+)
def find(items: list[str], target: str) -> str | None:
    for item in items:
        if item == target:
            return item
    return None

print(find(['a', 'b', 'c'], 'b'))
print(find(['a', 'b', 'c'], 'z'))

In [ ]:
# Optional[X] is equivalent to X | None (from typing, works on Python 3.8+)
from typing import Optional

def find(items: list[str], target: str) -> Optional[str]:
    return next((i for i in items if i == target), None)

find(['a', 'b', 'c'], 'b')

In [ ]:
# Union[X, Y]: accepts either type (Python 3.10+: X | Y)
def double(value: int | float) -> int | float:
    return value * 2

print(double(3))
print(double(2.5))

## 5. Callable

`Callable[[arg_types], return_type]` annotates a parameter that is itself a function.

In [ ]:
from typing import Callable

def apply(fn: Callable[[int], int], value: int) -> int:
    return fn(value)

apply(lambda x: x ** 2, 5)

In [ ]:
# Callable with multiple args
def run(fn: Callable[[str, int], str], name: str, n: int) -> str:
    return fn(name, n)

run(lambda s, n: s * n, 'ha', 3)

## 6. TypeVar: generic functions

`TypeVar` lets you write a function that preserves the type of its input without hard-coding what that type is.

In [ ]:
from typing import TypeVar

T = TypeVar('T')

def first(items: list[T]) -> T:
    return items[0]

print(first([1, 2, 3]))      # inferred as int
print(first(['a', 'b']))     # inferred as str

In [ ]:
# TypeVar with a bound (T must be a subtype of the bound)
from typing import TypeVar

Comparable = TypeVar('Comparable', int, float, str)

def clamp(value: Comparable, lo: Comparable, hi: Comparable) -> Comparable:
    return max(lo, min(hi, value))

print(clamp(15, 0, 10))
print(clamp(3.5, 0.0, 5.0))

## 7. Annotating Classes

Annotate instance variables in `__init__`, and use `ClassVar` for class-level variables.

In [ ]:
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner: str = owner
        self.balance: float = balance

    def deposit(self, amount: float) -> float:
        self.balance += amount
        return self.balance

acc = BankAccount('Alice', 100.0)
acc.deposit(50.0)

In [ ]:
from typing import ClassVar

class Config:
    # ClassVar marks a variable as belonging to the class, not instances
    MAX_CONNECTIONS: ClassVar[int] = 10

    def __init__(self, host: str, port: int) -> None:
        self.host = host
        self.port = port

print(Config.MAX_CONNECTIONS)
print(Config('localhost', 8080).host)

## 8. Protocol: structural subtyping

`Protocol` defines an interface by shape (the methods it has), not by inheritance. Any class with the right methods satisfies the protocol automatically, with no explicit subclassing needed.

In [ ]:
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> str:
        ...

def render(shape: Drawable) -> str:
    return shape.draw()

In [ ]:
# neither class inherits from Drawable; they satisfy it by having a draw() method
class Circle:
    def draw(self) -> str:
        return 'O'

class Square:
    def draw(self) -> str:
        return '[]'

print(render(Circle()))
print(render(Square()))

In [ ]:
# Protocol with multiple required methods
class Sized(Protocol):
    def __len__(self) -> int: ...
    def __contains__(self, item: object) -> bool: ...

def has_items(container: Sized) -> bool:
    return len(container) > 0

print(has_items([1, 2, 3]))
print(has_items([]))
print(has_items({'a': 1}))